## Settings

In [1]:
%pip install -q transformers peft nltk scikit-learn scipy pandas matplotlib pillow kaggle


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, subprocess, getpass

# GPU choice: all of this must happen BEFORE `import torch`
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"                        # GPU numbers match what nvidia-smi shows
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"    # less memory fragmentation
os.environ["TOKENIZERS_PARALLELISM"] = "false"                        # silences the "process just got forked" warnings
os.environ["USE_TF"] = "0"                                            # transformers: use PyTorch only

def pick_free_gpu():
    """Print the free memory of every GPU and return the id of the one with the most free memory."""
    try:
        rows = subprocess.check_output(["nvidia-smi", "--query-gpu=index,memory.used,memory.total",
                                        "--format=csv,noheader,nounits"], text=True).strip().splitlines()
    except Exception as e:
        print("could not run nvidia-smi:", e)
        return "0"
    gpus = []
    for r in rows:
        i, used, total = [int(x) for x in r.split(",")]
        gpus.append((i, total - used, total))
        print(f"GPU {i}: {(total - used) / 1024:5.1f} GiB free of {total / 1024:.1f} GiB")
    return str(max(gpus, key=lambda g: g[1])[0])

GPU_ID = pick_free_gpu()                      # or set it by hand, e.g. GPU_ID = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
print("using GPU", GPU_ID)

# ---- imports (torch only now, after the GPU is chosen) ----
import re, json, math, copy, pickle, random, string, textwrap, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU found! Check GPU_ID / nvidia-smi."
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
try:
    USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False)   # real bf16 only
except TypeError:                                                                              # older torch
    USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
print("Using:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (slow!)",
      "| torch", torch.__version__, "| bf16:", USE_BF16)

# SETTINGS
SMOKE_TEST = False              # True = quick trial run. Set to False for the real run.
ENCODER = "inception"          # "inception" or "vit"  (run the notebook once per encoder)
EPOCHS = 20                    # max epochs for the LSTM models; early stopping usually stops sooner
BATCH_SIZE = 256               # (image, prefix, next-word) training pairs per batch
LR = 1e-3
WORD_COUNT_THRESHOLD = 10      # drop words seen fewer than this many times in training captions
EMBED_DIM = 200                # GloVe 200d
UNITS = 256
DROPOUT = 0.5
N_EVAL = None                  # None = whole test set (1000 images). Use e.g. 300 to scale evaluation back
BEAM_WIDTHS = [3, 5, 7, 10]
TRAIN_INJECT_BASELINE = True   # also train a simple "inject" baseline
RUN_STRETCH_DECODING = True    # temperature / top-k / top-p / min-p / contrastive
RUN_LORA = True                # optional pretrained ViT-GPT-2 model fine-tuned with LoRA (Step 15)
RETRAIN = False                # True = ignore saved weights and train again
NUM_WORKERS = 4                # data-loading processes
SEED = 42
# LoRA settings
LORA_BASE = "nlpconnect/vit-gpt2-image-captioning"   # pretrained ViT encoder + GPT-2 decoder
LORA_R = 8                     # LoRA rank: size of the small trainable matrices
LORA_ALPHA = 16                # LoRA scaling
LORA_EPOCHS = 3
LORA_LR = 3e-4
LORA_BATCH = 8                 # images per step on the GPU (smaller = less memory)
LORA_ACCUM = 4                 # gradient accumulation: effective batch = LORA_BATCH * LORA_ACCUM = 32
LORA_MAX_STEPS = None          # None = train for LORA_EPOCHS full epochs (counted in optimizer steps)
#

if SMOKE_TEST:                 # small and fast: only checks that the pipeline works
    EPOCHS, N_EVAL, TRAIN_INJECT_BASELINE, RUN_STRETCH_DECODING = 3, 100, False, False
    LORA_EPOCHS, LORA_MAX_STEPS = 1, 30
    print(">>> SMOKE TEST MODE: tiny run, results are NOT meaningful. Set SMOKE_TEST = False for the real run.")
RUN_TAG = "_smoke" if SMOKE_TEST else ""     # keeps smoke-test weights/results separate from the real ones

DATA_DIR = Path("data")        # datasets are downloaded here
CACHE = "cache"                # features, trained weights and results are saved here
DATA_DIR.mkdir(exist_ok=True)
os.makedirs(CACHE, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

GPU 0:   9.9 GiB free of 11.0 GiB
GPU 1:  11.0 GiB free of 11.0 GiB
GPU 2:  11.0 GiB free of 11.0 GiB
GPU 3:  11.0 GiB free of 11.0 GiB
using GPU 1
Using: NVIDIA GeForce GTX 1080 Ti | torch 2.4.1+cu121 | bf16: False


In [3]:
token = getpass.getpass("Kaggle API token (press Enter to skip if you only have a kaggle.json): ")
if token.strip():
    os.environ["KAGGLE_API_TOKEN"] = token.strip()
else:
    user = input("Kaggle username (press Enter to skip): ").strip()
    if user:
        os.environ["KAGGLE_USERNAME"] = user
        os.environ["KAGGLE_KEY"] = getpass.getpass("Kaggle key: ").strip()
print("credentials set for this session" if any(k.startswith("KAGGLE_") for k in os.environ)
      else "no credentials entered (fine if you used `kaggle auth login`)")

Kaggle API token (press Enter to skip if you only have a kaggle.json):  ········


credentials set for this session


In [4]:
import zipfile, urllib.request

# Flickr8k
if not list(DATA_DIR.rglob("Flickr8k.token.txt")):
    from kaggle.api.kaggle_api_extended import KaggleApi    # imported here, after the credentials are set
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files("shadabhussain/flickr8k", path=str(DATA_DIR), unzip=True, quiet=False)
else:
    print("Flickr8k already downloaded")

#GloVe (keep only the 200d file)
glove_txt = DATA_DIR / "glove6b" / "glove.6B.200d.txt"
if not glove_txt.exists():
    zip_path = DATA_DIR / "glove.6B.zip"
    print("downloading GloVe (about 800 MB) ...")
    urllib.request.urlretrieve("https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip", zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extract("glove.6B.200d.txt", DATA_DIR / "glove6b")
    zip_path.unlink()
else:
    print("GloVe already downloaded")

print("token file:", list(DATA_DIR.rglob("Flickr8k.token.txt")))
print("glove file:", glove_txt, "| exists:", glove_txt.exists())

Flickr8k already downloaded
GloVe already downloaded
token file: [PosixPath('data/Flickr_Data/Flickr_Data/Flickr_TextData/Flickr8k.token.txt')]
glove file: data/glove6b/glove.6B.200d.txt | exists: True


## Captions, vocabulary and splits

In [5]:
found = sorted(DATA_DIR.rglob("Flickr8k.token.txt"))
if not found:
    raise FileNotFoundError("Flickr8k.token.txt not found under 'data/'. Run the download cell above first.")
BASE = str(found[0].parent.parent)              # the folder that contains Flickr_TextData and Images
token_path = f"{BASE}/Flickr_TextData/Flickr8k.token.txt"
train_list = f"{BASE}/Flickr_TextData/Flickr_8k.trainImages.txt"
dev_list   = f"{BASE}/Flickr_TextData/Flickr_8k.devImages.txt"
test_list  = f"{BASE}/Flickr_TextData/Flickr_8k.testImages.txt"
images_dir = f"{BASE}/Images/"
glove_dir  = str(DATA_DIR / "glove6b")
img_path = lambda image_id: os.path.join(images_dir, image_id + ".jpg")

def load_ids(path):
    with open(path) as f:
        return [line.strip().split(".")[0] for line in f if len(line.strip()) > 1]

train_ids, test_ids = load_ids(train_list), load_ids(test_list)
if os.path.exists(dev_list):
    val_ids = load_ids(dev_list)                 # official dev split, used for early stopping
else:
    val_ids, train_ids = train_ids[-500:], train_ids[:-500]

# --- parse + clean captions (lowercase, strip punctuation) ---
punct = str.maketrans("", "", string.punctuation)
descriptions = {}                                # image id -> list of cleaned caption strings
with open(token_path) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        image_id = parts[0].split(".")[0]
        words = [w.lower().translate(punct) for w in parts[1:]]
        descriptions.setdefault(image_id, []).append(" ".join(w for w in words if w))

def with_markers(ids):
    return {i: ["startseq " + c + " endseq" for c in descriptions[i]] for i in ids if i in descriptions}

train_descriptions = with_markers(train_ids)
val_descriptions = with_markers(val_ids)

# --- vocabulary: keep words seen >= WORD_COUNT_THRESHOLD times in the training captions ---
word_counts = {}
for caps in train_descriptions.values():
    for c in caps:
        for w in c.split():
            word_counts[w] = word_counts.get(w, 0) + 1
vocab = [w for w, n in word_counts.items() if n >= WORD_COUNT_THRESHOLD]
wordtoix = {w: i + 1 for i, w in enumerate(vocab)}      # index 0 is reserved for padding
ixtoword = {i: w for w, i in wordtoix.items()}
vocab_size = len(wordtoix) + 1
START, END = wordtoix["startseq"], wordtoix["endseq"]
max_length = max(len(c.split()) for caps in train_descriptions.values() for c in caps)

# human reference captions (as token lists) for evaluation
refs_tokens = {i: [c.split() for c in descriptions[i]] for i in test_ids if i in descriptions}

print(f"images  train/val/test: {len(train_ids)}/{len(val_ids)}/{len(test_ids)}")
print(f"words in all captions: {len({w for caps in descriptions.values() for c in caps for w in c.split()})} | vocab kept: {len(vocab)}")
print(f"max caption length (with markers): {max_length}")

images  train/val/test: 6000/1000/1000
words in all captions: 8828 | vocab kept: 1659
max caption length (with markers): 38


## Image features

In [6]:
class ImageFileDataset(torch.utils.data.Dataset):
    def __init__(self, paths, transform):
        self.paths, self.transform = paths, transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, k):
        from PIL import Image
        return self.transform(Image.open(self.paths[k]).convert("RGB"))

@torch.no_grad()
def extract_inception(paths, batch_size=64):
    from torchvision.models import inception_v3, Inception_V3_Weights
    weights = Inception_V3_Weights.IMAGENET1K_V1
    net = inception_v3(weights=weights)
    net.fc = nn.Identity()                                   # drop the 1000-way classifier -> 2048-d features
    net = net.to(DEVICE).eval()
    dl = torch.utils.data.DataLoader(ImageFileDataset(paths, weights.transforms()), batch_size=batch_size, num_workers=NUM_WORKERS)
    out = []
    for i, x in enumerate(dl):
        out.append(net(x.to(DEVICE)).cpu().numpy())
        if i % 20 == 0:
            print(f"Inception features: {min((i + 1) * batch_size, len(paths))}/{len(paths)}")
    return np.concatenate(out)

@torch.no_grad()
def extract_vit(paths, batch_size=32, name="google/vit-base-patch16-224-in21k"):
    from PIL import Image
    from transformers import ViTImageProcessor, ViTModel
    proc = ViTImageProcessor.from_pretrained(name)
    vit = ViTModel.from_pretrained(name, add_pooling_layer=False).to(DEVICE).eval()
    out = []
    for i in range(0, len(paths), batch_size):
        imgs = [Image.open(p).convert("RGB") for p in paths[i:i + batch_size]]
        inputs = proc(images=imgs, return_tensors="pt").to(DEVICE)
        out.append(vit(**inputs).last_hidden_state[:, 0].cpu().numpy())      # CLS token
        if (i // batch_size) % 20 == 0:
            print(f"ViT features: {i}/{len(paths)}")
    return np.concatenate(out)

In [7]:
def get_features(enc_name):
    cache_file = os.path.join(CACHE, f"feats_{enc_name}.pkl")
    if os.path.exists(cache_file):
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    ids = train_ids + val_ids + test_ids
    paths = [img_path(i) for i in ids]
    if enc_name == "inception":
        mat = extract_inception(paths)
    elif enc_name == "vit":
        mat = extract_vit(paths)
    else:
        raise ValueError("ENCODER must be 'inception' or 'vit'")
    feats = {i: mat[k].astype("float32") for k, i in enumerate(ids)}
    with open(cache_file, "wb") as f:
        pickle.dump(feats, f)
    return feats

FEATS_RAW = get_features(ENCODER)
tr = np.stack([FEATS_RAW[i] for i in train_ids])
mu, sd = tr.mean(0), np.maximum(tr.std(0), 1e-3)
FEATS = {i: ((v - mu) / sd).astype("float32") for i, v in FEATS_RAW.items()}
FEAT_DIM = tr.shape[1]
all_ids = train_ids + val_ids + test_ids
id2row = {i: k for k, i in enumerate(all_ids)}
feat_matrix = np.stack([FEATS[i] for i in all_ids]).astype("float32")
feat_gpu = torch.tensor(feat_matrix, device=DEVICE)
print(f"encoder = {ENCODER} | feature dim = {FEAT_DIM} | images = {len(all_ids)}")

encoder = inception | feature dim = 2048 | images = 8000


## GloVe word vectors

In [8]:
embeddings_index = {}
with open(os.path.join(glove_dir, "glove.6B.200d.txt"), encoding="utf-8") as f:
    for line in f:
        w, rest = line.split(" ", 1)
        if w in wordtoix:                       # only keep the words we need
            embeddings_index[w] = np.asarray(rest.split(), dtype="float32")

embedding_matrix = np.zeros((vocab_size, EMBED_DIM), dtype="float32")
for w, i in wordtoix.items():
    if w in embeddings_index:
        embedding_matrix[i] = embeddings_index[w]
print(f"GloVe covers {len(embeddings_index)}/{len(wordtoix)} vocab words (the rest start at zero; startseq/endseq are expected misses)")

GloVe covers 1650/1659 vocab words (the rest start at zero; startseq/endseq are expected misses)


## Models and training

### 7a. Training pairs

In [9]:
def make_pairs(descs):
    img_idx, seqs, targets = [], [], []
    for key, caps in descs.items():
        row = id2row[key]
        for cap in caps:
            seq = [wordtoix[w] for w in cap.split() if w in wordtoix]
            for i in range(1, len(seq)):
                img_idx.append(row); seqs.append(seq[:i]); targets.append(seq[i])
    X = np.zeros((len(seqs), max_length), dtype=np.int64)
    lens = np.zeros(len(seqs), dtype=np.int64)
    for k, s in enumerate(seqs):
        X[k, :len(s)] = s; lens[k] = len(s)
    to_gpu = lambda a: torch.from_numpy(a).to(DEVICE)
    return to_gpu(np.array(img_idx, dtype=np.int64)), to_gpu(X), to_gpu(lens), to_gpu(np.array(targets, dtype=np.int64))

def batches(pairs, batch_size, shuffle):
    img_idx, X, lens, y = pairs
    n = len(y)
    order = torch.randperm(n, device=DEVICE) if shuffle else torch.arange(n, device=DEVICE)
    for i in range(0, n, batch_size):
        b = order[i:i + batch_size]
        yield feat_gpu[img_idx[b]], X[b], lens[b], y[b]

train_pairs, val_pairs = make_pairs(train_descriptions), make_pairs(val_descriptions)
print(f"training pairs: {len(train_pairs[3]):,} | validation pairs: {len(val_pairs[3]):,}")

training pairs: 339,484 | validation pairs: 56,766


### 7b. Model definitions and the training loop

In [10]:
class MergeCaptioner(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.img_drop = nn.Dropout(DROPOUT)
        self.img_fc = nn.Linear(feat_dim, UNITS)
        self.emb = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix), freeze=True, padding_idx=0)   # frozen GloVe
        self.txt_drop = nn.Dropout(DROPOUT)
        self.lstm = nn.LSTM(EMBED_DIM, UNITS, batch_first=True)
        self.merge_fc = nn.Linear(2 * UNITS, UNITS)
        self.out = nn.Linear(UNITS, vocab_size)

    def forward(self, feats, seqs, lens):
        fe = F.relu(self.img_fc(self.img_drop(feats)))
        hidden, _ = self.lstm(self.txt_drop(self.emb(seqs)))
        se = hidden[torch.arange(len(seqs), device=seqs.device), lens - 1]       # output at the last real word
        return self.out(F.relu(self.merge_fc(torch.cat([fe, se], dim=1))))        # scores (logits) for every word

class InjectCaptioner(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.img_drop = nn.Dropout(DROPOUT)
        self.init_h = nn.Linear(feat_dim, UNITS)                                   # image -> initial LSTM state
        self.init_c = nn.Linear(feat_dim, UNITS)
        self.emb = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix), freeze=True, padding_idx=0)
        self.txt_drop = nn.Dropout(DROPOUT)
        self.lstm = nn.LSTM(EMBED_DIM, UNITS, batch_first=True)
        self.out_drop = nn.Dropout(DROPOUT)
        self.out = nn.Linear(UNITS, vocab_size)

    def forward(self, feats, seqs, lens):
        x = self.img_drop(feats)
        h0, c0 = torch.tanh(self.init_h(x)).unsqueeze(0), torch.tanh(self.init_c(x)).unsqueeze(0)
        hidden, _ = self.lstm(self.txt_drop(self.emb(seqs)), (h0, c0))
        se = hidden[torch.arange(len(seqs), device=seqs.device), lens - 1]
        return self.out(self.out_drop(se))

def run_epoch(model, pairs, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total, count = torch.zeros((), device=DEVICE), 0
    with torch.set_grad_enabled(training):
        for f, s, l, y in batches(pairs, BATCH_SIZE, shuffle=training):
            loss = F.cross_entropy(model(f, s, l), y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total += loss.detach() * len(y); count += len(y)
    return (total / count).item()

def fit_or_load(model, name, patience=3):
    ckpt = os.path.join(CACHE, f"{name}_{ENCODER}{RUN_TAG}.pt")
    hist_path = ckpt.replace(".pt", "_history.json")
    if os.path.exists(ckpt) and not RETRAIN:
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print(f"loaded saved weights for {name}")
        if os.path.exists(hist_path):
            with open(hist_path) as f:
                return json.load(f)
        print(f"  (no saved loss history for {name}: delete {ckpt} or set RETRAIN = True once to get the curves)")
        return None
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
    hist = {"loss": [], "val_loss": []}
    best, best_state, bad = float("inf"), None, 0
    for epoch in range(EPOCHS):
        t0 = time.time()
        tr_loss, va_loss = run_epoch(model, train_pairs, opt), run_epoch(model, val_pairs)
        hist["loss"].append(tr_loss); hist["val_loss"].append(va_loss)
        print(f"{name} epoch {epoch + 1:2d}/{EPOCHS} | train {tr_loss:.3f} | val {va_loss:.3f} | {time.time() - t0:.0f}s")
        if va_loss < best - 1e-4:
            best, best_state, bad = va_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                print("early stopping"); break
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    torch.save(best_state, ckpt)
    with open(hist_path, "w") as f:
        json.dump(hist, f)
    return hist

### 7c. Train and plot

In [11]:
merge_model = MergeCaptioner(FEAT_DIM).to(DEVICE)
print("merge model trainable parameters:", sum(p.numel() for p in merge_model.parameters() if p.requires_grad))
hist_merge = fit_or_load(merge_model, "merge")

inject_model, hist_inject = None, None
if TRAIN_INJECT_BASELINE:
    inject_model = InjectCaptioner(FEAT_DIM).to(DEVICE)
    hist_inject = fit_or_load(inject_model, "inject")

curves = [(n, h) for n, h in [("merge", hist_merge), ("inject", hist_inject)] if h is not None]
if curves:
    plt.figure(figsize=(6, 3.2))
    for name, h in curves:
        plt.plot(h["loss"], label=f"{name} train")
        plt.plot(h["val_loss"], "--", label=f"{name} val")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Training curves"); plt.tight_layout(); plt.show()
else:
    print("No loss history to plot (weights were loaded from cache without a history file). Set RETRAIN = True once to train again and save it.")

merge model trainable parameters: 1551484
merge epoch  1/20 | train 3.543 | val 3.110 | 12s
merge epoch  2/20 | train 2.877 | val 2.986 | 12s
merge epoch  3/20 | train 2.678 | val 2.942 | 12s
merge epoch  4/20 | train 2.561 | val 2.957 | 12s
merge epoch  5/20 | train 2.480 | val 2.958 | 12s
merge epoch  6/20 | train 2.419 | val 2.996 | 12s
early stopping


FileNotFoundError: [Errno 2] No such file or directory: 'cache/merge_inception.pt'

## Decoding

In [ ]:
V = vocab_size

@torch.no_grad()
def predict_probs(model, feats, seqs, batch_size=2048):
    """Probability of every vocabulary word being the next word, for each (image, words-so-far) row."""
    model.eval()
    feats = torch.as_tensor(feats, dtype=torch.float32)
    seqs = torch.as_tensor(seqs, dtype=torch.long)
    out = []
    for i in range(0, len(seqs), batch_size):
        f, s = feats[i:i + batch_size].to(DEVICE), seqs[i:i + batch_size].to(DEVICE)
        lens = (s != 0).sum(1).clamp(min=1)
        out.append(F.softmax(model(f, s, lens).float(), dim=-1).cpu().numpy())
    return np.concatenate(out)

def ids_to_words(ids):
    words = []
    for i in ids[1:]:
        if i == 0 or i == END:
            break
        words.append(ixtoword[int(i)])
    return words

def _pack(seqs, logp, ntok):
    return {"caps": [ids_to_words(s) for s in seqs], "logp": np.asarray(logp, dtype=float), "ntok": np.asarray(ntok)}

# ---------------------------- greedy ----------------------------
def greedy_batch(model, feats, max_len=None):
    max_len = max_len or max_length
    N = len(feats)
    seqs = np.zeros((N, max_len), dtype=np.int32); seqs[:, 0] = START
    logp_sum, ntok, done = np.zeros(N), np.zeros(N, dtype=int), np.zeros(N, dtype=bool)
    for t in range(1, max_len):
        if done.all():
            break
        probs = predict_probs(model, feats, seqs).astype(np.float64)
        probs[:, 0] = 0; probs[:, START] = 0
        nxt = probs.argmax(1)
        lp = np.log(probs[np.arange(N), nxt] + 1e-12)
        active = ~done
        seqs[active, t] = nxt[active]; logp_sum[active] += lp[active]; ntok[active] += 1
        done |= (nxt == END)
    return _pack(seqs, logp_sum, ntok)

# ---------------------------- beam search ----------------------------
def beam_search_batch(model, feats, K=3, max_len=None, length_alpha=0.0):
    """Beam search for all images at once. Scores are summed log-probabilities.
    length_alpha=0 -> pick the beam with the best raw log-prob; >0 divides by length**alpha (length normalisation)."""
    max_len = max_len or max_length
    N = len(feats)
    seqs = np.zeros((N, K, max_len), dtype=np.int32); seqs[:, :, 0] = START
    scores = np.full((N, K), -1e9); scores[:, 0] = 0.0          # at t=0 only one beam is alive
    finished = np.zeros((N, K), dtype=bool)
    gen_len = np.zeros((N, K), dtype=np.int32)                   # generated tokens so far (incl. endseq)
    rows = np.arange(N)[:, None]
    for t in range(1, max_len):
        if finished.all():
            break
        probs = predict_probs(model, np.repeat(feats, K, axis=0), seqs.reshape(N * K, max_len))
        logp = np.log(probs.astype(np.float64) + 1e-12).reshape(N, K, V)
        logp[:, :, 0] = -1e9; logp[:, :, START] = -1e9
        cand = scores[:, :, None] + logp                         # (N, K, V)
        cand[finished] = -1e9                                    # an ended beam can only stay as it is ...
        cand[:, :, 0] = np.where(finished, scores, -1e9)         # ... via the padding column, score unchanged
        flat = cand.reshape(N, K * V)
        top = np.argpartition(-flat, K - 1, axis=1)[:, :K]
        top_val = np.take_along_axis(flat, top, 1)
        order = np.argsort(-top_val, axis=1)
        top, top_val = np.take_along_axis(top, order, 1), np.take_along_axis(top_val, order, 1)
        beam_idx, tok = top // V, top % V
        seqs = seqs[rows, beam_idx]
        prev_fin = finished[rows, beam_idx]
        gen_len = gen_len[rows, beam_idx] + (~prev_fin)
        seqs[:, :, t] = np.where(prev_fin, 0, tok)
        finished = prev_fin | (tok == END)
        scores = top_val
    norm = scores / np.maximum(gen_len, 1) ** length_alpha
    best = norm.argmax(1)
    pick = np.arange(N)
    return _pack(seqs[pick, best], scores[pick, best], gen_len[pick, best])

# ---------------------------- sampling family ----------------------------
def filter_probs(p, temperature=1.0, top_k=None, top_p=None, min_p=None):
    """p: (N, V), normalised. Returns the reshaped / truncated, renormalised distribution."""
    p = p.copy()
    if temperature != 1.0:
        p = np.power(p, 1.0 / temperature)
        p /= p.sum(1, keepdims=True)
    if min_p is not None:                                        # keep words with prob >= min_p * (best word's prob)
        p = np.where(p >= min_p * p.max(1, keepdims=True), p, 0.0)
    if top_k is not None:                                        # keep only the k most likely words
        kth = np.partition(p, -top_k, axis=1)[:, -top_k][:, None]
        p = np.where(p >= kth, p, 0.0)
    if top_p is not None:                                        # smallest set of words whose cumulative prob reaches top_p
        order = np.argsort(-p, axis=1)
        sp = np.take_along_axis(p, order, 1)
        keep_sorted = (np.cumsum(sp, 1) - sp) < top_p
        keep = np.zeros_like(p, dtype=bool)
        np.put_along_axis(keep, order, keep_sorted, 1)
        p = np.where(keep, p, 0.0)
    return p / p.sum(1, keepdims=True)

def sample_batch(model, feats, temperature=1.0, top_k=None, top_p=None, min_p=None, seed=0, max_len=None):
    max_len = max_len or max_length
    rng = np.random.default_rng(seed)
    N = len(feats)
    seqs = np.zeros((N, max_len), dtype=np.int32); seqs[:, 0] = START
    logp_sum, ntok, done = np.zeros(N), np.zeros(N, dtype=int), np.zeros(N, dtype=bool)
    for t in range(1, max_len):
        if done.all():
            break
        probs = predict_probs(model, feats, seqs).astype(np.float64)
        probs[:, 0] = 0; probs[:, START] = 0
        probs /= probs.sum(1, keepdims=True)
        filt = filter_probs(probs, temperature, top_k, top_p, min_p)
        cdf = filt.cumsum(1); cdf[:, -1] = 1.0
        nxt = np.minimum((cdf < rng.random((N, 1))).sum(1), V - 1)
        lp = np.log(probs[np.arange(N), nxt] + 1e-12)            # confidence under the ORIGINAL model distribution
        active = ~done
        seqs[active, t] = nxt[active]; logp_sum[active] += lp[active]; ntok[active] += 1
        done |= (nxt == END)
    return _pack(seqs, logp_sum, ntok)

# ---------------------------- contrastive search (simplified) ----------------------------
def contrastive_batch(model, feats, k=5, alpha=0.4, max_len=None):
    """Simplified: among the top-k words, score = (1-alpha)*p(word) - alpha*(max cosine similarity between the word's
    embedding and the embeddings of the words generated so far). Real contrastive search compares hidden states of the
    language model; our LSTM does not expose per-step states, so this is a token-level version."""
    max_len = max_len or max_length
    E = embedding_matrix / (np.linalg.norm(embedding_matrix, axis=1, keepdims=True) + 1e-9)
    N = len(feats)
    seqs = np.zeros((N, max_len), dtype=np.int32); seqs[:, 0] = START
    logp_sum, ntok, done = np.zeros(N), np.zeros(N, dtype=int), np.zeros(N, dtype=bool)
    for t in range(1, max_len):
        if done.all():
            break
        probs = predict_probs(model, feats, seqs).astype(np.float64)
        probs[:, 0] = 0; probs[:, START] = 0
        topk = np.argpartition(-probs, k - 1, axis=1)[:, :k]                    # (N, k)
        topk_p = np.take_along_axis(probs, topk, 1)
        sims = np.einsum("nkd,ntd->nkt", E[topk], E[seqs[:, :t]])               # (N, k, t)
        score = (1 - alpha) * topk_p - alpha * sims.max(2)
        nxt = topk[np.arange(N), score.argmax(1)]
        lp = np.log(probs[np.arange(N), nxt] + 1e-12)
        active = ~done
        seqs[active, t] = nxt[active]; logp_sum[active] += lp[active]; ntok[active] += 1
        done |= (nxt == END)
    return _pack(seqs, logp_sum, ntok)

## Metrics

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
from scipy.stats import spearmanr, wilcoxon
from sklearn.metrics import roc_auc_score

_smooth = SmoothingFunction().method1
BLEU_W = {"BLEU-1": (1, 0, 0, 0), "BLEU-2": (.5, .5, 0, 0), "BLEU-3": (1/3, 1/3, 1/3, 0), "BLEU-4": (.25, .25, .25, .25)}

def _nonempty(caps):
    return [c if c else ["<empty>"] for c in caps]

def bleu_corpus(caps, refs):
    caps = _nonempty(caps)
    return {k: corpus_bleu(refs, caps, weights=w) for k, w in BLEU_W.items()}

def bleu_sentence(caps, refs):
    caps = _nonempty(caps)
    s1 = np.array([sentence_bleu(r, h, weights=BLEU_W["BLEU-1"], smoothing_function=_smooth) for h, r in zip(caps, refs)])
    s4 = np.array([sentence_bleu(r, h, weights=BLEU_W["BLEU-4"], smoothing_function=_smooth) for h, r in zip(caps, refs)])
    return s1, s4

# hallucination proxy
OBJECT_GROUPS = {
    "person":   {"man","men","woman","women","boy","boys","girl","girls","child","children","kid","kids","person","people",
                 "player","players","baby","guy","lady","crowd","group","team","skier","surfer","biker","rider","climber","swimmer"},
    "dog":      {"dog","dogs","puppy","puppies"},
    "horse":    {"horse","horses","pony"},
    "bike":     {"bike","bikes","bicycle","motorcycle","motorbike","dirtbike"},
    "ball":     {"ball","balls","football","soccer","basketball","baseball","volleyball"},
    "water":    {"water","ocean","lake","river","pool","sea","waves","wave","surf","stream"},
    "snow":     {"snow","snowy","ski","skis","snowboard","slope"},
    "beach":    {"beach","sand","shore","sandy"},
    "car":      {"car","cars","truck","van","vehicle","bus"},
    "boat":     {"boat","boats","kayak","canoe","raft","ship","sailboat"},
    "grass":    {"grass","grassy","field","lawn"},
    "mountain": {"mountain","mountains","hill","hills","cliff","rock","rocks"},
    "building": {"building","house","church","tower"},
    "street":   {"street","road","sidewalk","city"},
}
WORD2GROUPS = {}
for g, ws in OBJECT_GROUPS.items():
    for w in ws:
        WORD2GROUPS.setdefault(w, set()).add(g)

def hallucination_stats(caps, refs):
    flag, n_obj, n_bad = [], [], []
    for c, rs in zip(caps, refs):
        ref_words = {w for r in rs for w in r}
        ref_groups = {g for g, ws in OBJECT_GROUPS.items() if ws & ref_words}
        hyp_groups = {g for w in c for g in WORD2GROUPS.get(w, ())}
        bad = hyp_groups - ref_groups
        flag.append(len(bad) > 0); n_obj.append(len(hyp_groups)); n_bad.append(len(bad))
    return np.array(flag), np.array(n_obj), np.array(n_bad)

# CLIP
CLIP_NAME = "openai/clip-vit-base-patch32"
CLIP_PREFIX = "A photo depicts "
_clip = {}

def _get_clip():
    if not _clip:
        import torch
        from transformers import CLIPModel, CLIPProcessor
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        _clip["model"] = CLIPModel.from_pretrained(CLIP_NAME).to(dev).eval()
        _clip["proc"] = CLIPProcessor.from_pretrained(CLIP_NAME)
        _clip["dev"] = dev
    return _clip

def _as_tensor(e):
    import torch
    if torch.is_tensor(e):
        return e
    pooled = getattr(e, "pooler_output", None)                   # newer transformers versions may return an output object
    return pooled if pooled is not None else e[0]

def clip_image_embeddings(ids, batch_size=32):
    import torch
    from PIL import Image
    c, out = _get_clip(), []
    for i in range(0, len(ids), batch_size):
        imgs = [Image.open(img_path(j)).convert("RGB") for j in ids[i:i + batch_size]]
        inp = c["proc"](images=imgs, return_tensors="pt").to(c["dev"])
        with torch.no_grad():
            e = _as_tensor(c["model"].get_image_features(**inp))
        out.append(torch.nn.functional.normalize(e, dim=-1).cpu().numpy())
    return np.concatenate(out)

def clip_text_embeddings(texts, batch_size=256):
    import torch
    c, out = _get_clip(), []
    for i in range(0, len(texts), batch_size):
        inp = c["proc"](text=texts[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=77).to(c["dev"])
        with torch.no_grad():
            e = _as_tensor(c["model"].get_text_features(**inp))
        out.append(torch.nn.functional.normalize(e, dim=-1).cpu().numpy())
    return np.concatenate(out)

def clipscore(texts, img_emb):
    t = clip_text_embeddings([CLIP_PREFIX + x for x in texts])
    cos = (img_emb * t).sum(1)
    return 2.5 * np.clip(cos, 0, None), cos

## Experiment 1, greedy vs. beam search

In [ ]:
eval_ids = test_ids[:N_EVAL] if N_EVAL else test_ids
eval_ids = [i for i in eval_ids if i in refs_tokens]
eval_feats = np.stack([FEATS[i] for i in eval_ids]).astype("float32")
eval_refs = [refs_tokens[i] for i in eval_ids]
clip_img = clip_image_embeddings(eval_ids)
print(f"evaluating on {len(eval_ids)} test images")

RUNS = {}
def score_run(name, out):
    caps = out["caps"]
    texts = [" ".join(c) for c in caps]
    s1, s4 = bleu_sentence(caps, eval_refs)
    cs, cos = clipscore(texts, clip_img)
    flag, n_obj, n_bad = hallucination_stats(caps, eval_refs)
    run = dict(name=name, caps=caps, texts=texts, s1=s1, s4=s4, clip=cs, clip_cos=cos,
               halluc_flag=flag.astype(float), n_obj=n_obj, n_bad=n_bad,
               avg_len=float(np.mean([len(c) for c in caps])), distinct=len(set(texts)) / len(texts),
               **bleu_corpus(caps, eval_refs))
    if "logp" in out:
        run.update(logp=out["logp"], ntok=out["ntok"], conf_mean=out["logp"] / np.maximum(out["ntok"], 1))
    RUNS[name] = run
    return run

def summary_table(names=None):
    rows = []
    for n, r in RUNS.items():
        if names is not None and n not in names:
            continue
        rows.append({"method": n, "BLEU-1": r["BLEU-1"], "BLEU-2": r["BLEU-2"], "BLEU-3": r["BLEU-3"], "BLEU-4": r["BLEU-4"],
                     "CLIPScore": r["clip"].mean(), "halluc. captions %": 100 * r["halluc_flag"].mean(),
                     "avg len": r["avg_len"], "distinct %": 100 * r["distinct"]})
    return pd.DataFrame(rows).set_index("method").round(3)

# retrieval baseline: caption of the most similar training image
train_mat = np.stack([FEATS[i] for i in train_ids])
a = eval_feats / (np.linalg.norm(eval_feats, axis=1, keepdims=True) + 1e-9)
b = train_mat / (np.linalg.norm(train_mat, axis=1, keepdims=True) + 1e-9)
nn_idx = (a @ b.T).argmax(1)
score_run("Retrieval baseline (nearest train image)", {"caps": [descriptions[train_ids[j]][0].split() for j in nn_idx]})

# inject baseline (greedy)
if inject_model is not None:
    score_run("Inject LSTM, greedy", greedy_batch(inject_model, eval_feats))

# main model: greedy + beam search
score_run("Merge, greedy", greedy_batch(merge_model, eval_feats))
for K in BEAM_WIDTHS:
    score_run(f"Merge, beam K={K}", beam_search_batch(merge_model, eval_feats, K=K))
K_LN = 5 if 5 in BEAM_WIDTHS else BEAM_WIDTHS[0]
score_run(f"Merge, beam K={K_LN} + length norm", beam_search_batch(merge_model, eval_feats, K=K_LN, length_alpha=1.0))

summary_table()

In [ ]:
# Is the beam-vs-greedy difference real? Paired comparison per image (Wilcoxon signed-rank test).
def paired(a_name, b_name, key):
    d = RUNS[a_name][key] - RUNS[b_name][key]
    try:
        p = wilcoxon(d).pvalue
    except ValueError:                    # all differences are exactly zero
        p = float("nan")
    return d.mean(), p

rows = []
for K in BEAM_WIDTHS:
    for label, key in [("sentence BLEU-1", "s1"), ("sentence BLEU-4", "s4"), ("CLIPScore", "clip"), ("hallucinated caption (0/1)", "halluc_flag")]:
        diff, p = paired(f"Merge, beam K={K}", "Merge, greedy", key)
        rows.append({"beam K": K, "metric": label, "mean diff (beam - greedy)": round(diff, 4), "p (Wilcoxon)": round(p, 4)})
pd.DataFrame(rows).pivot(index="beam K", columns="metric", values=["mean diff (beam - greedy)", "p (Wilcoxon)"])

In [ ]:
# How do the metrics move with beam width? (K=1 is greedy)
Ks = [1] + BEAM_WIDTHS
names = ["Merge, greedy"] + [f"Merge, beam K={K}" for K in BEAM_WIDTHS]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, (title, fn) in zip(axes, [("BLEU-4 (corpus)", lambda r: r["BLEU-4"]),
                                  ("CLIPScore (mean)", lambda r: r["clip"].mean()),
                                  ("captions with a hallucinated object (%)", lambda r: 100 * r["halluc_flag"].mean()),
                                  ("distinct captions (%)", lambda r: 100 * r["distinct"])]):
    ax.plot(Ks, [fn(RUNS[n]) for n in names], "o-")
    ax.set_xlabel("beam width K"); ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

## Experiment 2, does confidence track quality?

In [ ]:
def spearman_ci(x, y, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    n = len(x)
    rhos = [spearmanr(x[idx], y[idx])[0] for idx in (rng.integers(0, n, n) for _ in range(n_boot))]
    lo, hi = np.nanpercentile(rhos, [2.5, 97.5])
    return spearmanr(x, y)[0], lo, hi

def confidence_report(run_name, conf_key="conf_mean"):
    r = RUNS[run_name]
    conf = r[conf_key]
    rows = []
    for label, q in [("sentence BLEU-1", r["s1"]), ("sentence BLEU-4", r["s4"]), ("CLIPScore", r["clip"])]:
        rho, lo, hi = spearman_ci(conf, q)
        auc = roc_auc_score(q <= np.percentile(q, 25), -conf)
        rows.append({"vs. quality metric": label, "Spearman rho": rho, "95% CI": f"[{lo:.2f}, {hi:.2f}]", "AUROC (low conf -> bad)": auc})
    flag = r["halluc_flag"].astype(bool)
    if 0 < flag.sum() < len(flag):
        rows.append({"vs. quality metric": "hallucinated object (yes/no)", "Spearman rho": spearmanr(conf, -r["halluc_flag"])[0],
                     "95% CI": "-", "AUROC (low conf -> bad)": roc_auc_score(flag, -conf)})
    return pd.DataFrame(rows).set_index("vs. quality metric").round(3)

conf_runs = ["Merge, greedy"] + [f"Merge, beam K={K}" for K in BEAM_WIDTHS]
for name in conf_runs:
    print(f"\n=== {name}   (confidence = mean log-prob per word) ===")
    display(confidence_report(name))

In [ ]:
BEAM_FOR_CONF = f"Merge, beam K={5 if 5 in BEAM_WIDTHS else BEAM_WIDTHS[0]}"

def quintile_plot(run_name, conf_key="conf_mean"):
    r = RUNS[run_name]
    df = pd.DataFrame({"conf": r[conf_key], "BLEU-1": r["s1"], "BLEU-4": r["s4"], "CLIPScore": r["clip"], "halluc. %": 100 * r["halluc_flag"]})
    df["bin"] = pd.qcut(df["conf"].rank(method="first"), 5, labels=["Q1\nleast sure", "Q2", "Q3", "Q4", "Q5\nmost sure"])
    g = df.groupby("bin", observed=True)[["BLEU-1", "BLEU-4", "CLIPScore", "halluc. %"]].mean()
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
    for ax, col in zip(axes, g.columns):
        g[col].plot.bar(ax=ax, rot=0); ax.set_title(f"{col} by confidence quintile", fontsize=10); ax.set_xlabel("")
    plt.suptitle(f"{run_name}: if confidence tracked quality, bars would rise (or fall for halluc.) left to right", y=1.03, fontsize=10)
    plt.tight_layout(); plt.show()
    return g

quintile_plot(BEAM_FOR_CONF)

r = RUNS[BEAM_FOR_CONF]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].scatter(r["conf_mean"], r["s1"], s=8, alpha=.4); axes[0].set_xlabel("mean log-prob per word"); axes[0].set_ylabel("sentence BLEU-1")
axes[1].scatter(r["conf_mean"], r["clip"], s=8, alpha=.4); axes[1].set_xlabel("mean log-prob per word"); axes[1].set_ylabel("CLIPScore")
plt.suptitle(f"Confidence vs. quality: {BEAM_FOR_CONF}", fontsize=10); plt.tight_layout(); plt.show()

## Looking at captions

In [ ]:
def show_grid(idxs, run_names, ncols=4):
    idxs = list(idxs)
    if not idxs:
        print("nothing to show"); return
    nrows = math.ceil(len(idxs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 5.4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, k in zip(axes, idxs):
        ax.imshow(plt.imread(img_path(eval_ids[k]))); ax.axis("off")
        lines = [f"{rn}: {RUNS[rn]['texts'][k]}" for rn in run_names] + ["human: " + " ".join(eval_refs[k][0])]
        ax.set_title("\n".join(textwrap.fill(l, 46) for l in lines), fontsize=8, loc="left")
    for ax in axes[len(idxs):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

rng = np.random.default_rng(1)
sample_idx = rng.choice(len(eval_ids), size=min(8, len(eval_ids)), replace=False)
show_grid(sample_idx, ["Merge, greedy", BEAM_FOR_CONF, f"Merge, beam K={BEAM_WIDTHS[-1]}"])

In [ ]:
def confidently_wrong(run_name, n=8, conf_key="conf_mean"):
    r = RUNS[run_name]
    conf, q = r[conf_key], r["clip"]
    idx = np.where((conf >= np.percentile(conf, 70)) & (q <= np.percentile(q, 30)))[0]
    idx = idx[np.argsort(-conf[idx])][:n]                       # most confident first
    print(f"{len(idx)} shown | top-30% confidence AND bottom-30% CLIPScore, most confident first")
    show_grid(idx, [run_name, "Merge, greedy"])

confidently_wrong(BEAM_FOR_CONF)

In [ ]:
# Do mistakes cluster around certain objects? An image belongs to a category if >= 2 of its 5 human captions mention the word.
CATEGORIES = ["dog", "man", "woman", "boy", "girl", "child", "water", "snow", "beach", "ball", "bike", "horse", "grass", "mountain"]
def cat_members(word):
    return np.array([sum(word in r for r in rs) >= 2 for rs in eval_refs])

rows = []
for cat in CATEGORIES:
    m = cat_members(cat)
    if m.sum() < 15:
        continue
    row = {"category": cat, "n images": int(m.sum())}
    for rn, short in [("Merge, greedy", "greedy"), (BEAM_FOR_CONF, "beam")]:
        r = RUNS[rn]
        row[f"{short} BLEU-1"] = r["s1"][m].mean(); row[f"{short} CLIP"] = r["clip"][m].mean(); row[f"{short} halluc %"] = 100 * r["halluc_flag"][m].mean()
    rows.append(row)
if rows:
    cat_df = pd.DataFrame(rows).set_index("category").round(3).sort_values("beam CLIP")
    display(cat_df)
    cat_df[["greedy CLIP", "beam CLIP"]].plot.barh(figsize=(7, 0.45 * len(cat_df) + 1.5))
    plt.xlabel("mean CLIPScore"); plt.title("Per-category quality (lowest at the top)"); plt.tight_layout(); plt.show()
else:
    print("No category has 15+ images in this evaluation set. Use a larger N_EVAL (or lower the threshold above).")

## Experiment 3, is CLIPScore a useful metric?

In [ ]:
# 1) human captions vs. generated captions, as scored by CLIP
counts = [len(rs) for rs in eval_refs]
offsets = np.cumsum([0] + counts[:-1])
ref_texts = [" ".join(r) for rs in eval_refs for r in rs]
t_emb = clip_text_embeddings([CLIP_PREFIX + x for x in ref_texts])
cos_h = (np.repeat(clip_img, counts, axis=0) * t_emb).sum(1)
human_clip = 2.5 * np.clip(np.add.reduceat(cos_h, offsets) / np.array(counts), 0, None)

# 2) control: same captions, but paired with a different image
mismatch = np.roll(np.arange(len(eval_ids)), 1)
shuffled = {n: clipscore(RUNS[n]["texts"], clip_img[mismatch])[0].mean() for n in ["Merge, greedy", BEAM_FOR_CONF]}

print(f"human captions   CLIPScore: {human_clip.mean():.3f}")
for n in ["Merge, greedy", BEAM_FOR_CONF]:
    print(f"{n:<16} CLIPScore: {RUNS[n]['clip'].mean():.3f}   (same captions on the WRONG image: {shuffled[n]:.3f})")

# 3a) per-caption agreement with BLEU
rows = [{"run": n, "rho(CLIP, sent. BLEU-1)": spearmanr(r["clip"], r["s1"])[0], "rho(CLIP, sent. BLEU-4)": spearmanr(r["clip"], r["s4"])[0]}
        for n, r in RUNS.items()]
display(pd.DataFrame(rows).set_index("run").round(3))

# 3b) do the two metrics rank the *methods* the same way?
means = pd.DataFrame({"BLEU-4": {n: r["BLEU-4"] for n, r in RUNS.items()}, "CLIPScore": {n: r["clip"].mean() for n, r in RUNS.items()}})
means["BLEU-4 rank"] = means["BLEU-4"].rank(ascending=False); means["CLIP rank"] = means["CLIPScore"].rank(ascending=False)
display(means.round(3))
print("Spearman between the two method rankings: %.2f (n=%d methods)" % (spearmanr(means["BLEU-4"], means["CLIPScore"])[0], len(means)))

# 4) which metric flags hallucinated captions better? (AUROC of -metric predicting the flag)
rows = []
for n in ["Merge, greedy", BEAM_FOR_CONF, f"Merge, beam K={BEAM_WIDTHS[-1]}"]:
    r = RUNS[n]; flag = r["halluc_flag"].astype(bool)
    if 0 < flag.sum() < len(flag):
        rows.append({"run": n, "AUROC CLIPScore": roc_auc_score(flag, -r["clip"]), "AUROC sent. BLEU-1": roc_auc_score(flag, -r["s1"]),
                     "AUROC sent. BLEU-4": roc_auc_score(flag, -r["s4"]), "AUROC model confidence": roc_auc_score(flag, -r["conf_mean"])})
display(pd.DataFrame(rows).set_index("run").round(3))

## Stretch decoders

In [ ]:
STRETCH = {
    "Sampling, temperature 0.7": dict(kind="sample", temperature=0.7),
    "Sampling, temperature 1.0": dict(kind="sample", temperature=1.0),
    "Top-k sampling, k=10":      dict(kind="sample", top_k=10),
    "Top-p sampling, p=0.9":     dict(kind="sample", top_p=0.9),
    "Min-p sampling, p=0.1":     dict(kind="sample", min_p=0.1),
    "Contrastive (simplified), k=5, alpha=0.4": dict(kind="contrastive", k=5, alpha=0.4),
}
if RUN_STRETCH_DECODING:
    for name, cfg in STRETCH.items():
        cfg = dict(cfg); kind = cfg.pop("kind")
        out = sample_batch(merge_model, eval_feats, seed=0, **cfg) if kind == "sample" else contrastive_batch(merge_model, eval_feats, **cfg)
        score_run(name, out)
    ref_rows = ["Merge, greedy", BEAM_FOR_CONF] + list(STRETCH)
    display(summary_table(ref_rows))

In [ ]:
import gc, sys

def free_gpu_memory():
    """Release GPU memory held by finished or failed cells (a failed cell keeps its tensors alive via its traceback)."""
    sys.last_type = sys.last_value = sys.last_traceback = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB still held by this notebook")

In [ ]:
for name in ["train_pairs", "val_pairs", "feat_gpu"]:     # LSTM training is finished, give this memory back
    if name in globals():
        del globals()[name]
free_gpu_memory()

# LoRA

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class CaptionPairs(Dataset):
    """One item per (image, caption) pair."""
    def __init__(self, ids, descs):
        self.items = [(i, c) for i in ids if i in descs for c in descs[i]]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, k):
        i, c = self.items[k]
        return Image.open(img_path(i)).convert("RGB"), c

def load_lora_base():
    from transformers import VisionEncoderDecoderModel, AutoTokenizer, AutoImageProcessor
    model = VisionEncoderDecoderModel.from_pretrained(LORA_BASE)
    tok = AutoTokenizer.from_pretrained(LORA_BASE)
    proc = AutoImageProcessor.from_pretrained(LORA_BASE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    start_id = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    for cfg in (model.config, model.generation_config):
        cfg.pad_token_id, cfg.eos_token_id, cfg.decoder_start_token_id = tok.pad_token_id, tok.eos_token_id, start_id
    return model, tok, proc

def make_collate(tok, proc, max_len=32):
    def collate(batch):
        imgs, caps = zip(*batch)
        pixel_values = proc(images=list(imgs), return_tensors="pt").pixel_values
        enc = tok([c + tok.eos_token for c in caps], padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        labels = enc.input_ids.masked_fill(enc.attention_mask == 0, -100)       # padding is ignored by the loss
        return pixel_values, labels
    return collate

def train_lora(peft_model, tok, proc):
    peft_model.to(DEVICE)
    collate = make_collate(tok, proc)
    train_dl = DataLoader(CaptionPairs(train_ids, descriptions), batch_size=LORA_BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate, drop_last=True)
    val_dl = DataLoader(CaptionPairs(val_ids[:100], descriptions), batch_size=LORA_BATCH, num_workers=NUM_WORKERS, collate_fn=collate)
    opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=LORA_LR, weight_decay=0.01)
    total = LORA_MAX_STEPS or (LORA_EPOCHS * len(train_dl)) // LORA_ACCUM            # optimizer steps
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 50) * max(0.0, 1 - s / total))
    step = 0
    for epoch in range(LORA_EPOCHS):
        peft_model.train()
        opt.zero_grad(set_to_none=True)
        for i, (pixel_values, labels) in enumerate(train_dl):
            pixel_values, labels = pixel_values.to(DEVICE), labels.to(DEVICE)
            with torch.autocast(device_type=DEVICE, dtype=torch.bfloat16, enabled=USE_BF16):
                loss = peft_model(pixel_values=pixel_values, labels=labels).loss / LORA_ACCUM
            loss.backward()
            if (i + 1) % LORA_ACCUM == 0:
                opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
                if step == 1 or step % 50 == 0:
                    peak = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE == "cuda" else 0.0
                    print(f"LoRA step {step}/{total} | loss {loss.item() * LORA_ACCUM:.3f} | peak GPU memory {peak:.1f} GiB")
                if step >= total:
                    break
        peft_model.eval(); val_sum, val_n = 0.0, 0
        with torch.no_grad():
            for pixel_values, labels in val_dl:
                pixel_values, labels = pixel_values.to(DEVICE), labels.to(DEVICE)
                val_sum += peft_model(pixel_values=pixel_values, labels=labels).loss.item() * len(pixel_values)
                val_n += len(pixel_values)
        print(f"LoRA epoch {epoch + 1}/{LORA_EPOCHS} | validation loss {val_sum / max(val_n, 1):.3f}")
        if step >= total:
            break

def get_lora_model():
    from peft import LoraConfig, get_peft_model, PeftModel
    base, tok, proc = load_lora_base()
    adapter_dir = os.path.join(CACHE, f"lora_adapter{RUN_TAG}")
    if os.path.exists(adapter_dir) and not RETRAIN:
        peft_model = PeftModel.from_pretrained(base, adapter_dir)
        print("loaded saved LoRA adapter")
    else:
        cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.1, bias="none",
                         target_modules=["query", "value", "c_attn", "q_attn"])   # attention layers of the ViT and of GPT-2
        peft_model = get_peft_model(base, cfg)
        peft_model.print_trainable_parameters()
        train_lora(peft_model, tok, proc)
        peft_model.save_pretrained(adapter_dir)
    merged = peft_model.merge_and_unload().to(DEVICE).eval()      # fold the LoRA update into the weights
    return merged, tok, proc

if RUN_LORA:
    lora_model, lora_tok, lora_proc = get_lora_model()

In [ ]:
def encode_pixels(ids, proc, batch_size=64):
    chunks = []
    for i in range(0, len(ids), batch_size):
        imgs = [Image.open(img_path(j)).convert("RGB") for j in ids[i:i + batch_size]]
        chunks.append(proc(images=imgs, return_tensors="pt").pixel_values)
    return torch.cat(chunks)

@torch.no_grad()
def sequence_logprobs(pixel_values, seqs):
    """Log-probability the model gives to each finished caption (teacher-forced), independent of how it was decoded."""
    eos = lora_tok.eos_token_id
    logits = lora_model(pixel_values=pixel_values, decoder_input_ids=seqs[:, :-1]).logits.float()
    target = seqs[:, 1:]
    lp = torch.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)       # (batch, length)
    is_eos = target == eos
    first_eos = torch.where(is_eos.any(1), is_eos.float().argmax(1), torch.full_like(is_eos.sum(1), target.shape[1] - 1))
    mask = torch.arange(target.shape[1], device=seqs.device)[None, :] <= first_eos[:, None]   # up to and including the first eos
    return (lp * mask).sum(1).cpu().numpy(), mask.sum(1).cpu().numpy()

@torch.no_grad()
def decode_lora(gen_kwargs, batch_size=64, seed=0):
    torch.manual_seed(seed)
    caps, logps, ntoks = [], [], []
    for i in range(0, len(eval_pixels), batch_size):
        pv = eval_pixels[i:i + batch_size].to(DEVICE)
        seqs = lora_model.generate(pixel_values=pv, max_new_tokens=30, **gen_kwargs)
        lp, nt = sequence_logprobs(pv, seqs)
        caps += [t.split() for t in lora_tok.batch_decode(seqs, skip_special_tokens=True)]
        logps.append(lp); ntoks.append(nt)
    return {"caps": caps, "logp": np.concatenate(logps), "ntok": np.concatenate(ntoks)}

if RUN_LORA:
    eval_pixels = encode_pixels(eval_ids, lora_proc)
    lora_decoders = {"greedy": dict(do_sample=False, num_beams=1)}
    for K in BEAM_WIDTHS:
        lora_decoders[f"beam K={K}"] = dict(do_sample=False, num_beams=K, length_penalty=0.0, early_stopping=True)
    lora_decoders[f"beam K={K_LN} + length norm"] = dict(do_sample=False, num_beams=K_LN, length_penalty=1.0, early_stopping=True)
    if RUN_STRETCH_DECODING:
        lora_decoders.update({
            "sampling, temperature 0.7": dict(do_sample=True, num_beams=1, temperature=0.7, top_k=0),
            "sampling, temperature 1.0": dict(do_sample=True, num_beams=1, temperature=1.0, top_k=0),
            "top-k sampling, k=10":      dict(do_sample=True, num_beams=1, top_k=10),
            "top-p sampling, p=0.9":     dict(do_sample=True, num_beams=1, top_k=0, top_p=0.9),
            "min-p sampling, p=0.1":     dict(do_sample=True, num_beams=1, top_k=0, min_p=0.1),
        })
    for name, kw in lora_decoders.items():
        score_run(f"LoRA ViT-GPT2, {name}", decode_lora(kw))
        print("done:", name)

## LoRA comparison

In [ ]:
if RUN_LORA:
    lora_names = [n for n in RUNS if n.startswith("LoRA ViT-GPT2")]
    compare = ["Merge, greedy", BEAM_FOR_CONF] + lora_names
    display(summary_table(compare))
    for n in ["LoRA ViT-GPT2, greedy", f"LoRA ViT-GPT2, {BEAM_FOR_CONF.replace('Merge, ', '')}"]:
        print(f"\n=== {n}   (confidence = mean log-prob per token) ===")
        display(confidence_report(n))

## Beam-search probability

In [ ]:
# The proposal asks about the beam-search probability itself (the TOTAL log-probability of the caption).
# That number mostly measures caption length (every extra word multiplies in another probability < 1),
# so it is shown next to the length-normalised version used elsewhere in this notebook.
rows = []
for name in ["Merge, greedy"] + [f"Merge, beam K={K}" for K in BEAM_WIDTHS]:
    r = RUNS[name]
    rho_len = spearmanr(r["logp"], r["ntok"])[0]
    for label, key in [("total log-prob", "logp"), ("mean log-prob per word", "conf_mean")]:
        rows.append({"run": name, "confidence measure": label,
                     "rho with caption length": round(rho_len, 3) if key == "logp" else round(spearmanr(r[key], r["ntok"])[0], 3),
                     "rho with sent. BLEU-4": round(spearmanr(r[key], r["s4"])[0], 3),
                     "rho with CLIPScore": round(spearmanr(r[key], r["clip"])[0], 3)})
display(pd.DataFrame(rows).set_index(["run", "confidence measure"]))

## Which method is better?

In [ ]:
# One table per model family.
# '*' = the per-image difference to greedy is significant (paired Wilcoxon, p < 0.05). No multiple-comparison correction.
def scorecard(family="Merge"):
    names = [n for n in RUNS if n.startswith("LoRA") == (family == "LoRA")]
    base = "LoRA ViT-GPT2, greedy" if family == "LoRA" else "Merge, greedy"
    rows = []
    for n in names:
        r = RUNS[n]
        row = {"method": n, "BLEU-4": r["BLEU-4"], "CLIPScore": r["clip"].mean(), "halluc. %": 100 * r["halluc_flag"].mean(),
               "distinct %": 100 * r["distinct"], "avg len": r["avg_len"]}
        if n != base and base in RUNS and "conf_mean" in r:
            for label, key in [("BLEU-4 vs greedy", "s4"), ("CLIP vs greedy", "clip")]:
                diff, p = paired(n, base, key)
                row[label] = f"{diff:+.3f}{'*' if p < 0.05 else ''}"
        rows.append(row)
    df = pd.DataFrame(rows).set_index("method")
    for c in ["BLEU-4 vs greedy", "CLIP vs greedy"]:
        if c in df.columns:
            df[c] = df[c].fillna("-")
    ranks = pd.DataFrame({"a": df["BLEU-4"].rank(ascending=False), "b": df["CLIPScore"].rank(ascending=False),
                          "c": df["halluc. %"].rank(ascending=True)})
    df["mean rank (BLEU-4, CLIP, halluc.)"] = ranks.mean(axis=1)
    return df.round(3).sort_values("mean rank (BLEU-4, CLIP, halluc.)")

display(scorecard("Merge"))
if any(n.startswith("LoRA") for n in RUNS):
    display(scorecard("LoRA"))

In [ ]:
table = summary_table()
table.to_csv(os.path.join(CACHE, f"results_{ENCODER}{RUN_TAG}.csv"))
with open(os.path.join(CACHE, f"results_{ENCODER}{RUN_TAG}.json"), "w") as f:
    json.dump(table.reset_index().to_dict(orient="records"), f, indent=1)
print(f"saved results for encoder = {ENCODER} to {CACHE}")

available = {}
for e in ["inception", "vit"]:
    p = os.path.join(CACHE, f"results_{e}{RUN_TAG}.json")
    if os.path.exists(p):
        with open(p) as f:
            available[e] = pd.DataFrame(json.load(f)).set_index("method")

if len(available) == 2:
    keep = [n for n in ["Retrieval baseline (nearest train image)", "Inject LSTM, greedy", "Merge, greedy", BEAM_FOR_CONF] if n in available["inception"].index and n in available["vit"].index]
    cols = ["BLEU-1", "BLEU-4", "CLIPScore", "halluc. captions %"]
    display(pd.concat({e: df.loc[keep, cols] for e, df in available.items()}, axis=1))
else:
    print("Only one encoder has been run so far. Switch ENCODER at the top and run again to compare.")

In [ ]:
def df_to_md(df, fmt="{:.3f}"):
    cols = [df.index.name or ""] + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for idx, row in df.iterrows():
        cells = [str(idx)] + [fmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

print("### Table 1: all methods\n");                       print(df_to_md(summary_table()))
print("\n### Table 2: confidence vs. quality (beam K=5)\n"); print(df_to_md(confidence_report(BEAM_FOR_CONF)))
print("\n### Table 3: scorecard (LSTM family)\n");          print(df_to_md(scorecard("Merge")))
if any(n.startswith("LoRA") for n in RUNS):
    print("\n### Table 4: scorecard (LoRA family)\n");      print(df_to_md(scorecard("LoRA")))

In [ ]:
os.makedirs("figures", exist_ok=True)

def run_and_save(fn, name, *args, **kwargs):
    """Runs a plotting function and saves its figure instead of only showing it."""
    original_show = plt.show
    def show_and_save(*a, **k):
        plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")
        original_show(*a, **k)
    plt.show = show_and_save
    try:
        return fn(*args, **kwargs)
    finally:
        plt.show = original_show

def beam_width_plot():
    Ks = [1] + BEAM_WIDTHS
    names = ["Merge, greedy"] + [f"Merge, beam K={K}" for K in BEAM_WIDTHS]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
    for ax, (title, fn) in zip(axes, [("BLEU-4 (corpus)", lambda r: r["BLEU-4"]), ("CLIPScore (mean)", lambda r: r["clip"].mean()),
                                      ("captions with a hallucinated object (%)", lambda r: 100 * r["halluc_flag"].mean()),
                                      ("distinct captions (%)", lambda r: 100 * r["distinct"])]):
        ax.plot(Ks, [fn(RUNS[n]) for n in names], "o-"); ax.set_xlabel("beam width K"); ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.show()

run_and_save(beam_width_plot, "beam_width_metrics")
run_and_save(quintile_plot, "confidence_quintiles", BEAM_FOR_CONF)
run_and_save(show_grid, "sample_captions", sample_idx, ["Merge, greedy", BEAM_FOR_CONF, f"Merge, beam K={BEAM_WIDTHS[-1]}"])
run_and_save(confidently_wrong, "confidently_wrong", BEAM_FOR_CONF)
print("saved:", sorted(os.listdir("figures")))